# Cloud Chaser Kaggle Notebook

This notebook recreates the full Cloud Chaser codebase inside a Kaggle working directory. Segmentation uses the Kaggle Sky-Image Dataset `swimseg-2` cloud masks, while cloud-type classification keeps GCD. Set Kaggle accelerator to GPU before running training cells.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/kaggle/working/cloud-chaser')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

for directory in ['cloud_chaser', 'cloud_chaser/data', 'cloud_chaser/models', 'cloud_chaser/utils', 'configs', 'scripts']:
    Path(directory).mkdir(parents=True, exist_ok=True)

print('Working in', PROJECT_DIR)


## Write Project Files

These cells materialize the Python package and CLI files.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "cloud-chaser"
version = "0.1.0"
description = "Two-stage cloud detection, segmentation, and meteorological classification."
readme = "README.md"
requires-python = ">=3.10,<3.13"
dependencies = [
  "albumentations>=1.4.0",
  "numpy>=1.24",
  "opencv-python>=4.8",
  "pillow>=10.0",
  "pyyaml>=6.0",
  "scikit-learn>=1.3",
  "torch>=2.1",
  "torchvision>=0.16",
  "tqdm>=4.66",
  "ultralytics>=8.2.0",
]

[project.optional-dependencies]
dev = ["ruff>=0.5", "pytest>=8.0"]

[project.scripts]
cloud-chaser-train = "train:main"
cloud-chaser-infer = "inference:main"
cloud-chaser-export = "export:main"

[tool.setuptools.packages.find]
include = ["cloud_chaser*"]

[tool.setuptools]
py-modules = ["train", "inference", "export"]

[tool.ruff]
line-length = 100
target-version = "py310"


In [ ]:
%%writefile requirements.txt
albumentations>=1.4.0
numpy>=1.24
opencv-python>=4.8
pillow>=10.0
pyyaml>=6.0
scikit-learn>=1.3
torch>=2.1
torchvision>=0.16
tqdm>=4.66
ultralytics>=8.2.0


In [ ]:
%%writefile README.md
# Cloud Chaser

## Abstract

Cloud Chaser is a two-stage computer vision system for cloud identification in unconstrained outdoor imagery. The system first localizes cloud-like regions using an instance segmentation model trained on environmental scene annotations, then classifies each detected cloud region into a meteorological category using a deep convolutional classifier. The design separates geometric localization from cloud-type recognition, allowing the segmentation module to handle background clutter such as buildings, trees, mountains, and blue sky, while the classification module focuses on texture, shape, and spatial structure within extracted cloud regions.

## 1. Problem Definition

The objective is to process a raw outdoor image and produce pixel-level cloud masks, bounding boxes, confidence scores, and meteorological cloud-type labels. Formally, given an image \(I \in \mathbb{R}^{H \times W \times 3}\), the system predicts a set of cloud instances:

\[
\mathcal{P} = \{(M_i, B_i, c_i, s_i, p_i)\}_{i=1}^{N}
\]

where \(M_i\) is a binary segmentation mask, \(B_i\) is a bounding box, \(c_i\) is the predicted cloud type, \(s_i\) is the segmentation confidence, and \(p_i\) is the classification confidence.

The task is challenging because clouds have ambiguous boundaries, variable scale, high intra-class diversity, and frequent visual overlap with sky, haze, smoke, snow, bright buildings, and other outdoor structures.

## 2. Dataset Strategy

### 2.1 Localization Dataset: ADE20K

ADE20K is used to train the segmentation component because it contains general-scene semantic annotations. Rather than training only on isolated sky imagery, ADE20K exposes the model to realistic environmental context, including urban, rural, coastal, mountainous, and indoor/outdoor transition scenes.

The pipeline extracts the relevant foreground classes, primarily `sky` and, when available, `cloud`. In this repository’s ADE20K metadata, `sky` is available as class ID `3`, while `cloud` is not present in the 150-class mapping. Therefore, the converter is configurable and falls back to `sky` when a dedicated cloud label is unavailable. This produces a one-class YOLO segmentation dataset where the foreground class is treated as cloud/sky-region candidate material.

### 2.2 Classification Dataset: GCD

The GCD dataset provides labeled images for seven cloud categories:

1. Cumulus
2. Altocumulus
3. Cirrus
4. Clear sky
5. Stratocumulus
6. Cumulonimbus
7. Mixed cloud

The classifier is trained using standard train, validation, and test partitions. If no validation directory exists, a stratified validation subset is deterministically sampled from the training set.

### 2.3 Self-Supervised Dataset: TJNU

The unlabeled TJNU dataset is used for self-supervised representation learning. Since cloud morphology is strongly defined by texture, density, illumination, and spatial structure, self-supervised pretraining helps the encoder learn useful visual embeddings before supervised fine-tuning on the smaller labeled GCD set.

## 3. Data Preprocessing and Augmentation

### 3.1 ADE20K Mask Conversion

ADE20K semantic masks are converted into YOLO segmentation labels. For each ADE mask, pixels belonging to the selected class IDs are combined into a binary foreground mask. Connected components are extracted, filtered by minimum area, converted into polygon contours, normalized to image coordinates, and written in YOLO segmentation format.

This conversion enables fine-tuning YOLO-Seg models using pixel-level supervision while preserving instance-like connected regions.

### 3.2 Meteorology-Aware Augmentations

The classification and self-supervised pipelines use augmentations selected to preserve meteorological texture patterns while improving robustness:

- Random shadows simulate illumination changes and partial occlusions.
- Gaussian blur models atmospheric haze, lens softness, and motion blur.
- Horizontal flips preserve cloud texture statistics and increase viewpoint diversity.
- Conservative vertical flips are included because cloud texture is often orientation-tolerant, but the probability is kept lower than horizontal flips to avoid excessive physical distortion.
- Color jitter accounts for exposure, white balance, and time-of-day variation.

## 4. Segmentation Module

### 4.1 Architecture

The segmentation module uses an Ultralytics YOLO segmentation architecture, configured by default as `yolo11s-seg.pt`. The model may also be switched to `yolov8s-seg.pt` through the YAML configuration.

YOLO-Seg is used because it provides:

- real-time or near-real-time detection speed,
- bounding boxes and masks from a single model,
- strong transfer learning from pretrained weights,
- straightforward export to ONNX and other deployment formats.

### 4.2 Training Objective

The detector is fine-tuned on the converted ADE20K masks. Its objective is to distinguish cloud/sky foreground from non-cloud background objects such as buildings, vegetation, roads, and mountains. The output consists of binary instance masks, bounding boxes, and confidence scores.

### 4.3 Output

For each detected instance, the detector returns:

- a bounding box \(B_i = (x_1, y_1, x_2, y_2)\),
- a pixel mask \(M_i\),
- a detection confidence \(s_i\).

These outputs are passed to the classification stage.

## 5. Feature Extraction and Classification Module

### 5.1 Two-Stage Recognition

The system uses a two-stage recognition strategy:

1. Localize candidate cloud regions with YOLO-Seg.
2. Classify each detected cloud crop using a CNN classifier.

This design is preferred over whole-image classification because a general outdoor image may contain many irrelevant objects. Mask-guided cropping reduces background bias and encourages the classifier to focus on cloud morphology.

### 5.2 Backbone Choices

The classifier supports the following convolutional backbones:

- ResNet50
- EfficientNet-B0
- DenseNet121

ResNet50 is the default because it offers a strong balance between representational capacity, training stability, and deployment compatibility.

### 5.3 Self-Supervised Pretraining

The implemented self-supervised method is SimCLR. Given an unlabeled cloud image, two augmented views are generated and passed through a shared encoder and projection head. The NT-Xent contrastive loss pulls embeddings from the same image together while pushing embeddings from different images apart.

This produces a cloud-aware encoder that can learn visual cues such as:

- fibrous cirrus texture,
- dense cumulonimbus vertical structure,
- stratocumulus layering,
- isolated cumulus contours,
- mixed cloud heterogeneity.

After pretraining, the encoder weights are transferred into the supervised classifier and fine-tuned on GCD labels.

### 5.4 Supervised Fine-Tuning

The classification head is trained with cross-entropy loss over the seven GCD classes. The training pipeline supports optional backbone freezing for early epochs, which stabilizes fine-tuning when the classifier is initialized from self-supervised weights.

## 6. Inference Pipeline

At inference time, the system processes a high-resolution image through the following steps:

1. Read the raw RGB image.
2. Run YOLO-Seg to detect cloud instances.
3. Resize masks to the original image resolution when necessary.
4. Use each mask and bounding box to extract a cloud-focused crop.
5. Batch all crops and pass them through the classifier.
6. Convert logits into class probabilities using softmax.
7. Overlay masks, bounding boxes, class names, and confidence scores on the original image.

The final output is an annotated image and a structured list of predictions.

## 7. Optimization and Deployment

The implementation supports several production-oriented optimizations:

- mixed precision training and inference on CUDA,
- batched classification of detected crops,
- configurable YOLO confidence and IoU thresholds,
- ONNX export for detector and classifier,
- TorchScript export for classifier deployment,
- centralized YAML configuration for reproducible experiments.

The modular code structure separates data processing, model definitions, training logic, evaluation, inference, visualization, and export.

## 8. Evaluation Metrics

### 8.1 Segmentation Metrics

The segmentation module is evaluated using:

- mask mAP, reported by Ultralytics validation,
- mask mAP@50,
- mean Intersection over Union over ADE20K validation foreground masks.

For binary mIoU, the predicted foreground mask is compared with the target cloud/sky mask:

\[
\text{IoU} = \frac{|M_{pred} \cap M_{gt}|}{|M_{pred} \cup M_{gt}|}
\]

### 8.2 Classification Metrics

The classifier is evaluated using:

- Top-1 accuracy,
- macro F1-score.

Macro F1 is important because cloud datasets may be class-imbalanced, and average accuracy alone can obscure weak performance on rarer cloud types such as cumulonimbus.

## 9. Reproducibility

The pipeline uses deterministic train/validation splitting for GCD and a centralized configuration file. Important experimental parameters are stored in `configs/default.yaml`, including dataset roots, image size, model backbones, learning rates, batch sizes, epoch counts, augmentation probabilities, detector thresholds, and checkpoint paths.

## 10. Limitations

The current ADE20K segmentation supervision depends on available semantic classes. If a dataset provides explicit cloud masks, those annotations should be preferred over generic sky masks. In the current ADE20K mapping, `cloud` is absent, so `sky` is used as a proxy for cloud-region awareness. This can cause the detector to include clear sky areas, which the classification stage partially addresses through the `Clear sky` class.

The classification model also depends on the quality and representativeness of GCD labels. Ambiguous cloud scenes, transitional cloud forms, and multi-layer atmospheres may be difficult to assign to a single class.

## 11. Conclusion

Cloud Chaser implements a practical two-stage approach for cloud identification in general outdoor imagery. By combining ADE20K-based environmental segmentation, self-supervised cloud representation learning, and supervised GCD classification, the system balances localization robustness with meteorological specificity. The resulting pipeline is suitable for experimentation, deployment-oriented optimization, and further research into cloud-type recognition under real-world visual conditions.


In [ ]:
%%writefile train.py
from __future__ import annotations

import argparse

from cloud_chaser.config import load_config
from cloud_chaser.evaluation import evaluate_detector
from cloud_chaser.training import evaluate_classifier, train_classifier, train_detector, train_ssl


def main() -> None:
    parser = argparse.ArgumentParser(description="Cloud Chaser training entrypoint")
    parser.add_argument("stage", choices=["detector", "ssl", "classifier", "eval-classifier", "eval-detector"])
    parser.add_argument("--config", default="configs/default.yaml")
    args = parser.parse_args()

    cfg = load_config(args.config)
    if args.stage == "detector":
        train_detector(cfg)
    elif args.stage == "ssl":
        train_ssl(cfg)
    elif args.stage == "classifier":
        train_classifier(cfg)
    elif args.stage == "eval-classifier":
        evaluate_classifier(cfg)
    elif args.stage == "eval-detector":
        evaluate_detector(cfg)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile inference.py
from __future__ import annotations

import argparse
from pathlib import Path

import cv2

from cloud_chaser.config import get_device, load_config
from cloud_chaser.inference_pipeline import CloudIdentifier


def main() -> None:
    parser = argparse.ArgumentParser(description="Run cloud segmentation and type classification.")
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--image", required=True)
    parser.add_argument("--output", default=None)
    args = parser.parse_args()

    cfg = load_config(args.config)
    data_cfg = cfg["data"]
    inf_cfg = cfg["inference"]
    det_cfg = cfg["detector"]
    identifier = CloudIdentifier(
        detector_weights=inf_cfg["detector_weights"],
        classifier_weights=inf_cfg["classifier_weights"],
        device=get_device(cfg),
        image_size=data_cfg["image_size"],
        detector_conf=det_cfg["conf"],
        detector_iou=det_cfg["iou"],
        half=det_cfg["half"],
        crop_padding=inf_cfg["crop_padding"],
    )
    overlay, predictions = identifier.predict(args.image)
    output = Path(args.output) if args.output else Path(inf_cfg["output_dir"]) / Path(args.image).name
    output.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output), overlay)
    for pred in predictions:
        print(
            f"{pred.class_name}: class={pred.class_confidence:.3f} "
            f"detector={pred.detector_confidence:.3f} box={pred.box}"
        )
    print(f"saved={output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export.py
from __future__ import annotations

import argparse
from pathlib import Path

import torch

from cloud_chaser.config import get_device, load_config
from cloud_chaser.models.classifier import CloudClassifier
from cloud_chaser.utils.checkpoint import load_checkpoint


def export_detector(weights: str, fmt: str, imgsz: int, half: bool) -> None:
    from ultralytics import YOLO

    YOLO(weights).export(format=fmt, imgsz=imgsz, half=half)


def export_classifier(cfg: dict, fmt: str, output: str | None = None) -> Path:
    device = get_device(cfg)
    checkpoint_path = cfg["classifier"]["checkpoint"]
    checkpoint = load_checkpoint(checkpoint_path, map_location=device)
    model = CloudClassifier(
        num_classes=len(checkpoint["classes"]),
        backbone=checkpoint["backbone"],
        dropout=0.0,
        pretrained=False,
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    image_size = cfg["data"]["image_size"]
    dummy = torch.randn(1, 3, image_size, image_size, device=device)
    output_path = Path(output or Path(checkpoint_path).with_suffix(f".{fmt}"))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if fmt == "torchscript":
        traced = torch.jit.trace(model, dummy)
        traced.save(str(output_path))
    elif fmt == "onnx":
        torch.onnx.export(
            model,
            dummy,
            str(output_path),
            input_names=["image"],
            output_names=["logits"],
            dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
            opset_version=17,
        )
    else:
        raise ValueError("Classifier format must be 'torchscript' or 'onnx'")
    return output_path


def main() -> None:
    parser = argparse.ArgumentParser(description="Export detector or classifier artifacts.")
    parser.add_argument("target", choices=["detector", "classifier"])
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--format", default="onnx", choices=["onnx", "torchscript", "engine"])
    parser.add_argument("--output", default=None)
    args = parser.parse_args()

    cfg = load_config(args.config)
    if args.target == "detector":
        export_detector(
            cfg["inference"]["detector_weights"],
            args.format,
            cfg["detector"]["imgsz"],
            cfg["detector"]["half"],
        )
    else:
        output = export_classifier(cfg, args.format, args.output)
        print(f"saved={output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile cloud_chaser/__init__.py
"""Cloud Chaser: cloud segmentation and meteorological type classification."""

__all__ = ["__version__"]
__version__ = "0.1.0"


In [ ]:
%%writefile cloud_chaser/config.py
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
from typing import Any

import yaml


def _deep_update(base: dict[str, Any], override: dict[str, Any]) -> dict[str, Any]:
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            base[key] = _deep_update(base[key], value)
        else:
            base[key] = value
    return base


def load_config(path: str | Path = "configs/default.yaml", overrides: dict[str, Any] | None = None) -> dict[str, Any]:
    """Load a YAML config and optionally apply nested dictionary overrides."""
    with Path(path).open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if overrides:
        cfg = _deep_update(deepcopy(cfg), overrides)
    return cfg


def save_yaml(data: dict[str, Any], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False)


def get_device(cfg: dict[str, Any]) -> str:
    import torch

    requested = cfg.get("project", {}).get("device", "cuda")
    if requested == "cuda" and not torch.cuda.is_available():
        return "cpu"
    return requested


In [ ]:
%%writefile cloud_chaser/data/__init__.py
"""Dataset and preprocessing utilities."""


In [ ]:
%%writefile cloud_chaser/data/ade20k.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm

from cloud_chaser.config import save_yaml


@dataclass(frozen=True)
class AdeClassMap:
    requested_names: list[str]
    class_ids: list[int]
    missing_names: list[str]


def parse_object_info(path: str | Path) -> dict[str, int]:
    mapping: dict[str, int] = {}
    with Path(path).open("r", encoding="utf-8") as f:
        next(f, None)
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 4:
                continue
            idx = int(parts[0].strip())
            names = [n.strip().lower() for n in parts[3].split(",") if n.strip()]
            for name in names:
                mapping[name] = idx
    return mapping


def resolve_ade_classes(
    root: str | Path,
    class_names: Iterable[str],
    fallback_ids: Iterable[int] | None = None,
) -> AdeClassMap:
    requested = [name.lower().strip() for name in class_names]
    info_path = Path(root) / "objectInfo150.txt"
    name_to_id = parse_object_info(info_path)
    class_ids: list[int] = []
    missing: list[str] = []
    for name in requested:
        if name in name_to_id:
            class_ids.append(name_to_id[name])
        else:
            missing.append(name)
    for idx in fallback_ids or []:
        if idx not in class_ids:
            class_ids.append(int(idx))
    if not class_ids:
        raise ValueError(
            f"None of the requested ADE classes {requested} were found in {info_path}, "
            "and no fallback IDs were supplied."
        )
    return AdeClassMap(requested_names=requested, class_ids=sorted(set(class_ids)), missing_names=missing)


def _iter_pairs(root: Path, split: str) -> list[tuple[Path, Path]]:
    image_dir = root / "images" / split
    mask_dir = root / "annotations" / split
    pairs: list[tuple[Path, Path]] = []
    for mask_path in sorted(mask_dir.glob("*.png")):
        stem = mask_path.stem
        image_path = image_dir / f"{stem}.jpg"
        if image_path.exists():
            pairs.append((image_path, mask_path))
    return pairs


def _mask_to_yolo_polygons(mask: np.ndarray, min_area: int) -> list[list[float]]:
    num_labels, labels = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)
    polygons: list[list[float]] = []
    h, w = mask.shape[:2]
    for component_id in range(1, num_labels):
        component = (labels == component_id).astype(np.uint8)
        if int(component.sum()) < min_area:
            continue
        contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < min_area or len(contour) < 3:
                continue
            epsilon = 0.0025 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True).reshape(-1, 2)
            if len(approx) < 3:
                continue
            coords: list[float] = []
            for x, y in approx:
                coords.extend([float(np.clip(x / w, 0, 1)), float(np.clip(y / h, 0, 1))])
            if len(coords) >= 6:
                polygons.append(coords)
    return polygons


def prepare_ade20k_yolo(
    root: str | Path,
    output_dir: str | Path,
    class_names: Iterable[str],
    fallback_class_ids: Iterable[int] | None = None,
    min_mask_area: int = 512,
) -> Path:
    """Convert ADE20K semantic masks into a one-class YOLO segmentation dataset."""
    root = Path(root)
    output_dir = Path(output_dir)
    class_map = resolve_ade_classes(root, class_names, fallback_class_ids)
    split_map = {"training": "train", "validation": "val"}

    for ade_split, yolo_split in split_map.items():
        image_out = output_dir / "images" / yolo_split
        label_out = output_dir / "labels" / yolo_split
        image_out.mkdir(parents=True, exist_ok=True)
        label_out.mkdir(parents=True, exist_ok=True)

        pairs = _iter_pairs(root, ade_split)
        for image_path, mask_path in tqdm(pairs, desc=f"Preparing ADE20K {ade_split}"):
            ade_mask = np.array(Image.open(mask_path))
            binary = np.isin(ade_mask, class_map.class_ids)
            polygons = _mask_to_yolo_polygons(binary, min_area=min_mask_area)
            if not polygons:
                continue
            target_image = image_out / image_path.name
            if not target_image.exists():
                target_image.symlink_to(image_path.resolve())
            label_path = label_out / f"{image_path.stem}.txt"
            lines = ["0 " + " ".join(f"{v:.6f}" for v in polygon) for polygon in polygons]
            label_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

    data_yaml = {
        "path": str(output_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "names": {0: "cloud"},
        "metadata": {
            "ade_class_ids": class_map.class_ids,
            "requested_ade_classes": class_map.requested_names,
            "missing_requested_classes": class_map.missing_names,
        },
    }
    save_yaml(data_yaml, output_dir / "cloud_seg.yaml")
    return output_dir / "cloud_seg.yaml"


In [ ]:
%%writefile cloud_chaser/data/augmentations.py
from __future__ import annotations

from typing import Any

import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def random_resized_crop(image_size: int, scale: tuple[float, float], ratio: tuple[float, float]):
    """Return RandomResizedCrop for both Albumentations 1.x and 2.x APIs."""
    try:
        return A.RandomResizedCrop(size=(image_size, image_size), scale=scale, ratio=ratio)
    except Exception:
        return A.RandomResizedCrop(height=image_size, width=image_size, scale=scale, ratio=ratio)


def classification_train_transforms(
    image_size: int,
    random_shadow_p: float = 0.25,
    gaussian_blur_p: float = 0.2,
    hflip_p: float = 0.5,
    vflip_p: float = 0.15,
) -> A.Compose:
    return A.Compose(
        [
            random_resized_crop(image_size, scale=(0.65, 1.0), ratio=(0.85, 1.2)),
            A.HorizontalFlip(p=hflip_p),
            A.VerticalFlip(p=vflip_p),
            A.RandomShadow(p=random_shadow_p),
            A.GaussianBlur(blur_limit=(3, 5), p=gaussian_blur_p),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.12, hue=0.03, p=0.35),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


def eval_transforms(image_size: int) -> A.Compose:
    return A.Compose(
        [
            A.Resize(image_size, image_size),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


def ssl_transforms(image_size: int, aug_cfg: dict[str, Any]) -> A.Compose:
    return A.Compose(
        [
            random_resized_crop(image_size, scale=(0.45, 1.0), ratio=(0.75, 1.33)),
            A.HorizontalFlip(p=aug_cfg.get("hflip_p", 0.5)),
            A.VerticalFlip(p=aug_cfg.get("vflip_p", 0.15)),
            A.RandomShadow(p=aug_cfg.get("random_shadow_p", 0.25)),
            A.GaussianBlur(blur_limit=(3, 7), p=aug_cfg.get("gaussian_blur_p", 0.2)),
            A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.04, p=0.55),
            A.ToGray(p=0.05),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


In [ ]:
%%writefile cloud_chaser/data/gcd.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_ALIASES = {
    "train": ("train", "training", "Train", "Training"),
    "val": ("val", "valid", "validation", "Val", "Valid", "Validation"),
    "test": ("test", "testing", "Test", "Testing"),
}
CLASS_KEYWORDS = ("cumulus", "altocumulus", "cirrus", "clearsky", "clear", "stratocumulus", "cumulonimbus", "mixed")
FORBIDDEN_CLASSIFIER_ROOTS = ("swimseg", "skyimage", "sky-image", "segmentation", "mask")


@dataclass(frozen=True)
class ImageRecord:
    path: Path
    label: int


def _is_forbidden_classifier_root(path: Path) -> bool:
    text = str(path).lower()
    return any(token in text for token in FORBIDDEN_CLASSIFIER_ROOTS)


def _has_images(path: Path) -> bool:
    return path.exists() and any(p.suffix.lower() in IMAGE_EXTENSIONS for p in path.rglob("*"))


def _image_count(path: Path) -> int:
    return sum(1 for p in path.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)


def _find_split_dir(root: Path, split: str) -> Path | None:
    for name in SPLIT_ALIASES[split]:
        candidate = root / name
        if candidate.exists() and candidate.is_dir():
            return candidate
    return None


def _class_dirs(path: Path) -> list[Path]:
    if not path.exists() or not path.is_dir():
        return []
    return sorted(p for p in path.iterdir() if p.is_dir() and _has_images(p))


def _looks_like_class_root(path: Path) -> bool:
    dirs = _class_dirs(path)
    if len(dirs) < 2:
        return False
    names = " ".join(d.name.lower().replace("_", "") for d in dirs)
    keyword_hits = sum(1 for keyword in CLASS_KEYWORDS if keyword in names)
    return keyword_hits >= 2 or len(dirs) >= 5


def _candidate_roots(root: Path) -> list[Path]:
    candidates: list[Path] = []
    if root.exists():
        candidates.append(root)
        candidates.extend(p for p in root.rglob("*") if p.is_dir())
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(p for p in kaggle_input.rglob("*") if p.is_dir())
        candidates.extend(p for p in kaggle_input.glob("*") if p.is_dir())
    seen: set[Path] = set()
    unique: list[Path] = []
    for path in candidates:
        if path not in seen:
            unique.append(path)
            seen.add(path)
    return unique


def resolve_gcd_root(root: str | Path) -> Path:
    root = Path(root)
    scored: list[tuple[int, int, Path]] = []
    for path in _candidate_roots(root):
        if _is_forbidden_classifier_root(path):
            continue
        train_dir = _find_split_dir(path, "train")
        if train_dir is not None and _class_dirs(train_dir):
            scored.append((_image_count(train_dir), 0, path))
        elif _looks_like_class_root(path):
            scored.append((_image_count(path), 1, path))
    if scored:
        return sorted(scored, key=lambda item: (-item[0], item[1], len(item[2].parts), str(item[2])))[0][2]

    available = []
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.glob("*")):
            available.append(f"{path} images={_image_count(path)}")
    raise FileNotFoundError(
        "Could not find GCD classification images. Attach the GCD dataset or set data.gcd_root to a folder "
        "containing either train/<class>/*.jpg or <class>/*.jpg. Available inputs: " + "; ".join(available)
    )


def discover_classes(root: str | Path) -> list[str]:
    root = resolve_gcd_root(root)
    train_root = _find_split_dir(root, "train") or root
    classes = sorted(p.name for p in _class_dirs(train_root))
    if not classes:
        raise RuntimeError(f"No class folders with images found under {train_root}")
    return classes


def _records_for_split(split_dir: Path | None, classes: list[str]) -> list[ImageRecord]:
    if split_dir is None or not split_dir.exists():
        return []
    records: list[ImageRecord] = []
    class_to_idx = {name: idx for idx, name in enumerate(classes)}
    available = {p.name.lower(): p for p in split_dir.iterdir() if p.is_dir()}
    for class_name in classes:
        class_dir = split_dir / class_name
        if not class_dir.exists():
            class_dir = available.get(class_name.lower())
        if class_dir is None or not class_dir.exists():
            continue
        for path in sorted(class_dir.rglob("*")):
            if path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append(ImageRecord(path=path, label=class_to_idx[class_name]))
    return records


def _split_records(records: list[ImageRecord], split: str, val_fraction: float, seed: int) -> list[ImageRecord]:
    labels = [r.label for r in records]
    indices = list(range(len(records)))
    train_idx, holdout_idx = train_test_split(indices, test_size=val_fraction * 2, random_state=seed, stratify=labels)
    holdout_labels = [records[i].label for i in holdout_idx]
    val_idx, test_idx = train_test_split(holdout_idx, test_size=0.5, random_state=seed, stratify=holdout_labels)
    selected = {"train": train_idx, "val": val_idx, "test": test_idx}[split]
    return [records[i] for i in selected]


def build_gcd_records(
    root: str | Path,
    split: str,
    classes: list[str] | None = None,
    val_fraction: float = 0.15,
    seed: int = 42,
) -> tuple[list[ImageRecord], list[str]]:
    root = resolve_gcd_root(root)
    discovered = discover_classes(root)
    classes = classes or discovered

    train_dir = _find_split_dir(root, "train")
    probe_root = train_dir or root
    probe_records = _records_for_split(probe_root, classes)
    if not probe_records:
        print(f"Configured GCD classes {classes} did not match folders under {probe_root}.")
        print(f"Using discovered classes instead: {discovered}")
        classes = discovered

    if train_dir is None:
        all_records = _records_for_split(root, classes)
        if not all_records:
            raise RuntimeError(f"No GCD images found under class folders in {root}")
        return _split_records(all_records, split, val_fraction, seed), classes

    if split == "test":
        records = _records_for_split(_find_split_dir(root, "test"), classes)
        if records:
            return records, classes
        all_train_records = _records_for_split(train_dir, classes)
        return _split_records(all_train_records, "test", val_fraction, seed), classes

    val_dir = _find_split_dir(root, "val")
    if split == "val" and val_dir is not None:
        records = _records_for_split(val_dir, classes)
        if records:
            return records, classes

    train_records = _records_for_split(train_dir, classes)
    if not train_records:
        raise RuntimeError(f"No GCD train images found under {train_dir}. Check data.gcd_root.")
    labels = [r.label for r in train_records]
    indices = list(range(len(train_records)))
    train_idx, val_idx = train_test_split(indices, test_size=val_fraction, random_state=seed, stratify=labels)
    selected = train_idx if split == "train" else val_idx
    return [train_records[i] for i in selected], classes


class GCDDataset(Dataset):
    def __init__(
        self,
        root: str | Path,
        split: str,
        transform=None,
        classes: list[str] | None = None,
        val_fraction: float = 0.15,
        seed: int = 42,
    ) -> None:
        self.records, self.classes = build_gcd_records(root, split, classes, val_fraction, seed)
        self.transform = transform
        print(f"GCD {split}: {len(self.records)} images, classes={self.classes}")

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        record = self.records[index]
        image = cv2.cvtColor(cv2.imread(str(record.path)), cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)["image"]
        else:
            image = Image.open(record.path).convert("RGB")
        return image, record.label


In [ ]:
%%writefile cloud_chaser/data/tjnu.py
from __future__ import annotations

from pathlib import Path

import cv2
from torch.utils.data import Dataset

from cloud_chaser.data.gcd import IMAGE_EXTENSIONS


class UnlabeledCloudDataset(Dataset):
    """Unlabeled image dataset for SimCLR/MoCo-style pretraining."""

    def __init__(self, root: str | Path, transform) -> None:
        self.root = Path(root)
        if not self.root.exists():
            raise FileNotFoundError(f"Unlabeled dataset not found: {self.root}")
        self.paths = sorted(
            p for p in self.root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        )
        if not self.paths:
            raise RuntimeError(f"No images found under {self.root}")
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int):
        image = cv2.cvtColor(cv2.imread(str(self.paths[index])), cv2.COLOR_BGR2RGB)
        view1 = self.transform(image=image)["image"]
        view2 = self.transform(image=image)["image"]
        return view1, view2


In [ ]:
%%writefile cloud_chaser/data/swimseg.py
from __future__ import annotations

import csv
import re
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from cloud_chaser.config import save_yaml
from cloud_chaser.data.gcd import IMAGE_EXTENSIONS

MASK_TOKENS = (
    "mask", "masks", "gt", "gtmaps", "groundtruth", "ground_truth", "truth",
    "label", "labels", "annotation", "annotations", "segmentation", "binary",
)


@dataclass(frozen=True)
class SwimsegPair:
    image_path: Path
    mask_path: Path


def _is_probable_binary_mask(path: Path) -> bool:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return False
    sample = image[:: max(1, image.shape[0] // 256), :: max(1, image.shape[1] // 256)]
    unique = np.unique(sample)
    if len(unique) <= 8:
        return True
    low_high = ((sample < 16) | (sample > 239)).mean()
    return bool(low_high > 0.97)


def _looks_like_mask_path(path: Path) -> bool:
    joined = "/".join(part.lower() for part in path.parts)
    return any(token in joined for token in MASK_TOKENS) or _is_probable_binary_mask(path)


def _normal_stem(path: Path) -> str:
    stem = path.stem.lower()
    stem = re.sub(r"(_|-)?(mask|gt|gtmap|groundtruth|ground_truth|truth|label|annotation|segmentation|binary)$", "", stem)
    stem = re.sub(r"[^a-z0-9]+", "", stem)
    return stem


def discover_swimseg_pairs(root: str | Path) -> list[SwimsegPair]:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"SWIMSEG root does not exist: {root}")

    files = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not files:
        raise RuntimeError(f"No image files found under {root}")

    mask_set = {p for p in files if _looks_like_mask_path(p)}
    mask_files = sorted(mask_set)
    image_files = sorted(p for p in files if p not in mask_set)

    if not image_files or not mask_files:
        mask_set = {p for p in files if _is_probable_binary_mask(p)}
        mask_files = sorted(mask_set)
        image_files = sorted(p for p in files if p not in mask_set)

    masks_by_key: dict[str, list[Path]] = {}
    for mask in mask_files:
        masks_by_key.setdefault(_normal_stem(mask), []).append(mask)

    pairs: list[SwimsegPair] = []
    used_masks: set[Path] = set()
    for image in image_files:
        key = _normal_stem(image)
        candidates = [m for m in masks_by_key.get(key, []) if m not in used_masks]
        if candidates:
            mask = candidates[0]
            pairs.append(SwimsegPair(image, mask))
            used_masks.add(mask)

    if len(pairs) < min(len(image_files), len(mask_files)) * 0.5:
        pairs = []
        for image, mask in zip(sorted(image_files), sorted(mask_files), strict=False):
            img = cv2.imread(str(image), cv2.IMREAD_COLOR)
            msk = cv2.imread(str(mask), cv2.IMREAD_GRAYSCALE)
            if img is not None and msk is not None and img.shape[:2] == msk.shape[:2]:
                pairs.append(SwimsegPair(image, mask))

    if not pairs:
        raise RuntimeError(
            f"Could not pair SWIMSEG images and masks under {root}. "
            "Inspect the dataset tree and adjust data.swimseg_root."
        )
    print(f"Discovered {len(pairs)} SWIMSEG image/mask pairs under {root}")
    return pairs


def _mask_to_yolo_polygons(mask: np.ndarray, min_area: int) -> list[list[float]]:
    mask = mask.astype(np.uint8)
    num_labels, labels = cv2.connectedComponents(mask, connectivity=8)
    h, w = mask.shape[:2]
    polygons: list[list[float]] = []
    for component_id in range(1, num_labels):
        component = (labels == component_id).astype(np.uint8)
        if int(component.sum()) < min_area:
            continue
        contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            if cv2.contourArea(contour) < min_area or len(contour) < 3:
                continue
            epsilon = 0.0025 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True).reshape(-1, 2)
            if len(approx) < 3:
                continue
            coords: list[float] = []
            for x, y in approx:
                coords.extend([float(np.clip(x / w, 0, 1)), float(np.clip(y / h, 0, 1))])
            if len(coords) >= 6:
                polygons.append(coords)
    return polygons


def _binary_cloud_mask(mask_path: Path, invert: bool = False) -> np.ndarray:
    mask = np.array(Image.open(mask_path).convert("L"))
    binary = mask > 127
    if invert:
        binary = ~binary
    return binary.astype(np.uint8)


def _safe_link_or_copy(source: Path, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        return
    try:
        target.symlink_to(source.resolve())
    except OSError:
        image = cv2.imread(str(source), cv2.IMREAD_UNCHANGED)
        if image is None:
            raise FileNotFoundError(source)
        cv2.imwrite(str(target), image)


def prepare_swimseg_yolo(
    root: str | Path,
    output_dir: str | Path,
    val_fraction: float = 0.1,
    test_fraction: float = 0.1,
    seed: int = 42,
    min_mask_area: int = 96,
    invert_masks: bool = False,
) -> Path:
    output_dir = Path(output_dir)
    pairs = discover_swimseg_pairs(root)
    indices = list(range(len(pairs)))
    train_idx, holdout_idx = train_test_split(indices, test_size=val_fraction + test_fraction, random_state=seed)
    relative_test = test_fraction / max(val_fraction + test_fraction, 1e-9)
    val_idx, test_idx = train_test_split(holdout_idx, test_size=relative_test, random_state=seed)
    split_indices = {"train": train_idx, "val": val_idx, "test": test_idx}

    manifest_path = output_dir / "manifest.csv"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["split", "image", "mask"])
        writer.writeheader()
        for split, split_idx in split_indices.items():
            for idx in tqdm(split_idx, desc=f"Preparing SWIMSEG {split}"):
                pair = pairs[idx]
                image = cv2.imread(str(pair.image_path), cv2.IMREAD_COLOR)
                if image is None:
                    continue
                h, w = image.shape[:2]
                binary = _binary_cloud_mask(pair.mask_path, invert=invert_masks)
                if binary.shape != (h, w):
                    binary = cv2.resize(binary, (w, h), interpolation=cv2.INTER_NEAREST)
                polygons = _mask_to_yolo_polygons(binary, min_area=min_mask_area)
                if not polygons:
                    continue

                target_image = output_dir / "images" / split / pair.image_path.name
                target_mask = output_dir / "masks" / split / f"{pair.image_path.stem}.png"
                label_path = output_dir / "labels" / split / f"{pair.image_path.stem}.txt"
                _safe_link_or_copy(pair.image_path, target_image)
                target_mask.parent.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(target_mask), binary * 255)
                label_path.parent.mkdir(parents=True, exist_ok=True)
                label_path.write_text(
                    "\n".join("0 " + " ".join(f"{v:.6f}" for v in poly) for poly in polygons) + "\n",
                    encoding="utf-8",
                )
                writer.writerow({"split": split, "image": str(target_image), "mask": str(target_mask)})

    data_yaml = {
        "path": str(output_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "cloud"},
        "metadata": {
            "source": "SWIMSEG-2 from Kaggle Sky-Image Dataset",
            "binary_mask_cloud_value": "white unless invert_masks=true",
            "pairs_discovered": len(pairs),
        },
    }
    save_yaml(data_yaml, output_dir / "cloud_seg.yaml")
    return output_dir / "cloud_seg.yaml"


In [ ]:
%%writefile cloud_chaser/evaluation.py
from __future__ import annotations

from pathlib import Path


def evaluate_detector(cfg: dict) -> dict[str, float]:
    """Evaluate YOLO mask mAP on the prepared SWIMSEG validation split.

    On Kaggle's Torch 2.10/Ultralytics stack, an additional per-image model.predict()
    mIoU loop can fail with "Inference tensors do not track version counter" after
    validation. Ultralytics val already reports mask mAP, so this evaluator avoids
    the fragile second prediction pass and keeps notebook runs resumable.
    """
    from ultralytics import YOLO

    data_cfg = cfg["data"]
    det_cfg = cfg["detector"]
    weights = cfg["inference"]["detector_weights"]
    model = YOLO(weights)
    data_yaml = Path(data_cfg["prepared_seg_dir"]) / "cloud_seg.yaml"
    map_metrics = model.val(
        data=str(data_yaml),
        task="segment",
        imgsz=det_cfg["imgsz"],
        conf=det_cfg["conf"],
        iou=det_cfg["iou"],
        half=det_cfg["half"],
        verbose=False,
    )
    metrics = {
        "mask_map50_95": float(getattr(map_metrics.seg, "map", 0.0)),
        "mask_map50": float(getattr(map_metrics.seg, "map50", 0.0)),
    }
    print(metrics)
    return metrics


In [ ]:
%%writefile cloud_chaser/inference_pipeline.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F

from cloud_chaser.data.augmentations import eval_transforms
from cloud_chaser.models.classifier import CloudClassifier
from cloud_chaser.utils.checkpoint import load_checkpoint
from cloud_chaser.utils.visualization import overlay_instance


def display_class_name(class_name: str) -> str:
    name = class_name.split("_", 1)[1] if "_" in class_name and class_name[0].isdigit() else class_name
    return name.replace("_", " ").title()


@dataclass
class CloudPrediction:
    box: tuple[int, int, int, int]
    detector_confidence: float
    class_name: str
    class_confidence: float


class CloudIdentifier:
    def __init__(
        self,
        detector_weights: str | Path,
        classifier_weights: str | Path,
        device: str = "cuda",
        image_size: int = 224,
        detector_conf: float = 0.25,
        detector_iou: float = 0.6,
        half: bool = True,
        crop_padding: int = 12,
    ) -> None:
        from ultralytics import YOLO

        self.device = device if device != "cuda" or torch.cuda.is_available() else "cpu"
        self.detector = YOLO(str(detector_weights))
        self.detector_conf = detector_conf
        self.detector_iou = detector_iou
        self.half = half and self.device != "cpu"
        self.crop_padding = crop_padding
        checkpoint = load_checkpoint(classifier_weights, map_location=self.device)
        self.classes = checkpoint["classes"]
        self.transform = eval_transforms(image_size)
        self.classifier = CloudClassifier(
            num_classes=len(self.classes),
            backbone=checkpoint["backbone"],
            dropout=0.0,
            pretrained=False,
        ).to(self.device)
        self.classifier.load_state_dict(checkpoint["model"])
        self.classifier.eval()

    def _crop_instances(
        self,
        image_rgb: np.ndarray,
        masks: torch.Tensor,
        boxes: torch.Tensor,
    ) -> list[torch.Tensor]:
        h, w = image_rgb.shape[:2]
        crops: list[torch.Tensor] = []
        for mask_tensor, box_tensor in zip(masks, boxes, strict=False):
            x1, y1, x2, y2 = [int(v) for v in box_tensor.tolist()]
            x1 = max(0, x1 - self.crop_padding)
            y1 = max(0, y1 - self.crop_padding)
            x2 = min(w, x2 + self.crop_padding)
            y2 = min(h, y2 + self.crop_padding)
            mask = mask_tensor.detach().cpu().numpy().astype(bool)
            masked = image_rgb.copy()
            masked[~mask] = 0
            crop = masked[y1:y2, x1:x2]
            crops.append(self.transform(image=crop)["image"])
        return crops

    @torch.inference_mode()
    def predict(self, image_path: str | Path) -> tuple[np.ndarray, list[CloudPrediction]]:
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            raise FileNotFoundError(f"Could not read image: {image_path}")
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        h, w = image_rgb.shape[:2]

        results = self.detector.predict(
            image_rgb,
            conf=self.detector_conf,
            iou=self.detector_iou,
            retina_masks=True,
            device=self.device,
            half=self.half,
            verbose=False,
        )
        result = results[0]
        if result.masks is None or result.boxes is None or len(result.boxes) == 0:
            return image_bgr, []

        masks = result.masks.data
        if masks.shape[-2:] != (h, w):
            masks = F.interpolate(masks[:, None].float(), size=(h, w), mode="nearest")[:, 0] > 0.5
        boxes = result.boxes.xyxy.detach().cpu()
        detector_scores = result.boxes.conf.detach().cpu().tolist()

        crops = self._crop_instances(image_rgb, masks, boxes)
        batch = torch.stack(crops).to(self.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.half):
            probs = torch.softmax(self.classifier(batch), dim=1)
        confs, labels = probs.max(dim=1)

        overlay = image_bgr.copy()
        predictions: list[CloudPrediction] = []
        for i, (box_tensor, det_score, label_idx, cls_score) in enumerate(
            zip(boxes, detector_scores, labels, confs, strict=False)
        ):
            box = tuple(int(v) for v in box_tensor.tolist())
            class_name = display_class_name(self.classes[int(label_idx)])
            class_conf = float(cls_score.detach().cpu())
            mask = masks[i].detach().cpu().numpy().astype(bool)
            overlay = overlay_instance(
                overlay,
                mask,
                box,
                class_name,
                class_conf,
                color_index=int(label_idx),
            )
            predictions.append(
                CloudPrediction(
                    box=box,
                    detector_confidence=float(det_score),
                    class_name=class_name,
                    class_confidence=class_conf,
                )
            )
        return overlay, predictions


In [ ]:
%%writefile cloud_chaser/models/__init__.py
"""Model definitions and wrappers."""


In [ ]:
%%writefile cloud_chaser/models/classifier.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

import torch
from torch import nn
from torchvision import models

BackboneName = Literal["resnet50", "efficientnet_b0", "densenet121"]


@dataclass(frozen=True)
class BackboneBundle:
    encoder: nn.Module
    features_dim: int


class DenseNetEncoder(nn.Module):
    def __init__(self, model: models.DenseNet) -> None:
        super().__init__()
        self.features = model.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.features(x))
        x = self.pool(x)
        return torch.flatten(x, 1)


def build_encoder(backbone: BackboneName = "resnet50", pretrained: bool = True) -> BackboneBundle:
    if backbone == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.resnet50(weights=weights)
        features_dim = model.fc.in_features
        model.fc = nn.Identity()
        return BackboneBundle(model, features_dim)
    if backbone == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        features_dim = model.classifier[1].in_features
        model.classifier = nn.Identity()
        return BackboneBundle(model, features_dim)
    if backbone == "densenet121":
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.densenet121(weights=weights)
        return BackboneBundle(DenseNetEncoder(model), model.classifier.in_features)
    raise ValueError(f"Unsupported backbone: {backbone}")


class CloudClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        backbone: BackboneName = "resnet50",
        dropout: float = 0.2,
        pretrained: bool = True,
    ) -> None:
        super().__init__()
        bundle = build_encoder(backbone, pretrained=pretrained)
        self.backbone_name = backbone
        self.encoder = bundle.encoder
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(bundle.features_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))

    def freeze_encoder(self, freeze: bool = True) -> None:
        for param in self.encoder.parameters():
            param.requires_grad = not freeze


In [ ]:
%%writefile cloud_chaser/models/detector.py
from __future__ import annotations

from pathlib import Path


def train_yolo_segmenter(
    model_name_or_path: str,
    data_yaml: str | Path,
    output_dir: str | Path,
    epochs: int,
    imgsz: int,
    batch: int,
    device: str,
    patience: int = 20,
    lr0: float = 0.01,
    weight_decay: float = 0.0005,
):
    from ultralytics import YOLO

    output_dir = Path(output_dir)
    yolo = YOLO(model_name_or_path)
    return yolo.train(
        data=str(data_yaml),
        task="segment",
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=device,
        project=str(output_dir.parent),
        name=output_dir.name,
        patience=patience,
        lr0=lr0,
        weight_decay=weight_decay,
        amp=True,
    )


In [ ]:
%%writefile cloud_chaser/models/simclr.py
from __future__ import annotations

import torch
from torch import nn
import torch.nn.functional as F

from cloud_chaser.models.classifier import BackboneName, build_encoder


class SimCLR(nn.Module):
    def __init__(
        self,
        backbone: BackboneName = "resnet50",
        projection_dim: int = 128,
        hidden_dim: int = 2048,
        pretrained: bool = True,
    ) -> None:
        super().__init__()
        bundle = build_encoder(backbone, pretrained=pretrained)
        self.backbone_name = backbone
        self.encoder = bundle.encoder
        self.projector = nn.Sequential(
            nn.Linear(bundle.features_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, projection_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.encoder(x)
        return F.normalize(self.projector(features), dim=1)


def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.2) -> torch.Tensor:
    batch_size = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    similarity = torch.matmul(z, z.T) / temperature
    mask = torch.eye(2 * batch_size, device=z.device, dtype=torch.bool)
    similarity = similarity.masked_fill(mask, -9e15)
    positives = torch.cat(
        [torch.arange(batch_size, 2 * batch_size), torch.arange(0, batch_size)]
    ).to(z.device)
    return F.cross_entropy(similarity, positives)


In [ ]:
%%writefile cloud_chaser/training.py
from __future__ import annotations

from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from cloud_chaser.config import get_device
from cloud_chaser.data.swimseg import prepare_swimseg_yolo
from cloud_chaser.data.augmentations import (
    classification_train_transforms,
    eval_transforms,
    ssl_transforms,
)
from cloud_chaser.data.gcd import GCDDataset
from cloud_chaser.data.tjnu import UnlabeledCloudDataset
from cloud_chaser.models.classifier import CloudClassifier
from cloud_chaser.models.detector import train_yolo_segmenter
from cloud_chaser.models.simclr import SimCLR, nt_xent_loss
from cloud_chaser.utils.checkpoint import load_checkpoint, save_checkpoint
from cloud_chaser.utils.metrics import classification_metrics
from cloud_chaser.utils.seed import seed_everything


def train_detector(cfg: dict) -> None:
    seed_everything(cfg["project"]["seed"])
    data_cfg = cfg["data"]
    detector_cfg = cfg["detector"]
    data_yaml = prepare_swimseg_yolo(
        root=data_cfg["swimseg_root"],
        output_dir=data_cfg["prepared_seg_dir"],
        val_fraction=data_cfg.get("seg_val_fraction", 0.1),
        test_fraction=data_cfg.get("seg_test_fraction", 0.1),
        seed=cfg["project"]["seed"],
        min_mask_area=data_cfg.get("min_mask_area", 96),
        invert_masks=data_cfg.get("swimseg_invert_masks", False),
    )
    output_dir = Path(cfg["project"]["output_dir"]) / "detector"
    train_yolo_segmenter(
        model_name_or_path=detector_cfg["model"],
        data_yaml=data_yaml,
        output_dir=output_dir,
        epochs=detector_cfg["epochs"],
        imgsz=detector_cfg["imgsz"],
        batch=detector_cfg["batch"],
        device=get_device(cfg),
        patience=detector_cfg["patience"],
        lr0=detector_cfg["lr0"],
        weight_decay=detector_cfg["weight_decay"],
    )


def train_ssl(cfg: dict) -> None:
    seed_everything(cfg["project"]["seed"])
    device = get_device(cfg)
    data_cfg = cfg["data"]
    ssl_cfg = cfg["ssl"]
    dataset = UnlabeledCloudDataset(
        data_cfg["tjnu_root"],
        ssl_transforms(data_cfg["image_size"], cfg["augmentation"]),
    )
    loader = DataLoader(
        dataset,
        batch_size=ssl_cfg["batch_size"],
        shuffle=True,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
        drop_last=True,
    )
    model = SimCLR(
        backbone=ssl_cfg["backbone"],
        projection_dim=ssl_cfg["projection_dim"],
        hidden_dim=ssl_cfg["hidden_dim"],
    ).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=ssl_cfg["lr"],
        weight_decay=ssl_cfg["weight_decay"],
    )
    scaler = torch.cuda.amp.GradScaler(enabled=device == "cuda")
    best_loss = float("inf")
    output_dir = Path(cfg["project"]["output_dir"]) / "ssl"

    for epoch in range(ssl_cfg["epochs"]):
        model.train()
        running_loss = 0.0
        for view1, view2 in tqdm(loader, desc=f"SSL epoch {epoch + 1}/{ssl_cfg['epochs']}"):
            view1 = view1.to(device, non_blocking=True)
            view2 = view2.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                loss = nt_xent_loss(model(view1), model(view2), ssl_cfg["temperature"])
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.detach().cpu())
        epoch_loss = running_loss / max(1, len(loader))
        save_checkpoint(
            {
                "epoch": epoch,
                "backbone": ssl_cfg["backbone"],
                "encoder": model.encoder.state_dict(),
                "model": model.state_dict(),
                "loss": epoch_loss,
            },
            output_dir / "last.pt",
        )
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            save_checkpoint(
                {
                    "epoch": epoch,
                    "backbone": ssl_cfg["backbone"],
                    "encoder": model.encoder.state_dict(),
                    "model": model.state_dict(),
                    "loss": epoch_loss,
                },
                output_dir / "best.pt",
            )


def _run_classifier_epoch(
    model: CloudClassifier,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
    optimizer: torch.optim.Optimizer | None = None,
    amp: bool = True,
) -> dict[str, float]:
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    all_logits: list[torch.Tensor] = []
    all_targets: list[torch.Tensor] = []
    scaler = torch.cuda.amp.GradScaler(enabled=training and amp and device == "cuda")
    for images, targets in tqdm(loader, desc="train" if training else "eval"):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.set_grad_enabled(training):
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp and device == "cuda"):
                logits = model(images)
                loss = criterion(logits, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += float(loss.detach().cpu())
        all_logits.append(logits.detach().cpu())
        all_targets.append(targets.detach().cpu())
    logits = torch.cat(all_logits)
    targets = torch.cat(all_targets)
    metrics = classification_metrics(logits, targets)
    metrics["loss"] = total_loss / max(1, len(loader))
    return metrics


def train_classifier(cfg: dict) -> None:
    seed_everything(cfg["project"]["seed"])
    device = get_device(cfg)
    data_cfg = cfg["data"]
    cls_cfg = cfg["classifier"]
    train_tfms = classification_train_transforms(data_cfg["image_size"], **cfg["augmentation"])
    eval_tfms = eval_transforms(data_cfg["image_size"])
    train_ds = GCDDataset(
        data_cfg["gcd_root"],
        "train",
        train_tfms,
        classes=data_cfg["classification_classes"],
        val_fraction=data_cfg["val_fraction"],
        seed=cfg["project"]["seed"],
    )
    val_ds = GCDDataset(
        data_cfg["gcd_root"],
        "val",
        eval_tfms,
        classes=train_ds.classes,
        val_fraction=data_cfg["val_fraction"],
        seed=cfg["project"]["seed"],
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=cls_cfg["batch_size"],
        shuffle=True,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cls_cfg["batch_size"],
        shuffle=False,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    model = CloudClassifier(
        num_classes=len(train_ds.classes),
        backbone=cls_cfg["backbone"],
        dropout=cls_cfg["dropout"],
    ).to(device)
    if cls_cfg.get("ssl_checkpoint"):
        checkpoint = load_checkpoint(cls_cfg["ssl_checkpoint"], map_location=device)
        model.encoder.load_state_dict(checkpoint["encoder"], strict=False)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cls_cfg["lr"], weight_decay=cls_cfg["weight_decay"])
    best_f1 = -1.0
    output_dir = Path(cfg["project"]["output_dir"]) / "classifier"

    for epoch in range(cls_cfg["epochs"]):
        model.freeze_encoder(epoch < cls_cfg.get("freeze_backbone_epochs", 0))
        train_metrics = _run_classifier_epoch(
            model, train_loader, criterion, device, optimizer=optimizer, amp=cls_cfg["amp"]
        )
        val_metrics = _run_classifier_epoch(model, val_loader, criterion, device, amp=cls_cfg["amp"])
        print(
            f"epoch={epoch + 1} train_loss={train_metrics['loss']:.4f} "
            f"val_top1={val_metrics['top1']:.4f} val_f1={val_metrics['f1_macro']:.4f}"
        )
        payload = {
            "epoch": epoch,
            "classes": train_ds.classes,
            "backbone": cls_cfg["backbone"],
            "model": model.state_dict(),
            "val_metrics": val_metrics,
        }
        save_checkpoint(payload, output_dir / "last.pt")
        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            save_checkpoint(payload, output_dir / "best.pt")


def evaluate_classifier(cfg: dict) -> dict[str, float]:
    device = get_device(cfg)
    data_cfg = cfg["data"]
    cls_cfg = cfg["classifier"]
    checkpoint = load_checkpoint(cls_cfg["checkpoint"], map_location=device)
    dataset = GCDDataset(
        data_cfg["gcd_root"],
        "test",
        eval_transforms(data_cfg["image_size"]),
        classes=checkpoint["classes"],
    )
    loader = DataLoader(
        dataset,
        batch_size=cls_cfg["batch_size"],
        shuffle=False,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    model = CloudClassifier(
        num_classes=len(checkpoint["classes"]),
        backbone=checkpoint["backbone"],
        dropout=0.0,
        pretrained=False,
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    metrics = _run_classifier_epoch(model, loader, nn.CrossEntropyLoss(), device, amp=cls_cfg["amp"])
    print(metrics)
    return metrics


In [ ]:
%%writefile cloud_chaser/utils/__init__.py
"""Shared utilities."""


In [ ]:
%%writefile cloud_chaser/utils/checkpoint.py
from __future__ import annotations

from pathlib import Path
from typing import Any

import torch


def save_checkpoint(payload: dict[str, Any], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


def load_checkpoint(path: str | Path, map_location: str | torch.device = "cpu") -> dict[str, Any]:
    return torch.load(Path(path), map_location=map_location)


In [ ]:
%%writefile cloud_chaser/utils/metrics.py
from __future__ import annotations

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score


def classification_metrics(logits: torch.Tensor, targets: torch.Tensor) -> dict[str, float]:
    preds = logits.argmax(dim=1).detach().cpu().numpy()
    labels = targets.detach().cpu().numpy()
    return {
        "top1": float(accuracy_score(labels, preds)),
        "f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0)),
    }


def binary_iou(pred: np.ndarray, target: np.ndarray) -> float:
    pred = pred.astype(bool)
    target = target.astype(bool)
    union = np.logical_or(pred, target).sum()
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, target).sum() / union)


In [ ]:
%%writefile cloud_chaser/utils/seed.py
from __future__ import annotations

import os
import random

import numpy as np
import torch


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True


In [ ]:
%%writefile cloud_chaser/utils/visualization.py
from __future__ import annotations

import cv2
import numpy as np


PALETTE = [
    (42, 157, 143),
    (233, 196, 106),
    (230, 111, 81),
    (38, 70, 83),
    (131, 197, 190),
    (239, 71, 111),
    (17, 138, 178),
]


def overlay_instance(
    image: np.ndarray,
    mask: np.ndarray,
    box: tuple[int, int, int, int],
    label: str,
    score: float,
    color_index: int,
    alpha: float = 0.42,
) -> np.ndarray:
    color = PALETTE[color_index % len(PALETTE)]
    out = image.copy()
    color_layer = np.zeros_like(out)
    color_layer[:, :] = color
    out[mask] = cv2.addWeighted(out, 1 - alpha, color_layer, alpha, 0)[mask]
    x1, y1, x2, y2 = box
    cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
    text = f"{label} - {score:.0%}"
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    y_text = max(0, y1 - th - 8)
    cv2.rectangle(out, (x1, y_text), (x1 + tw + 8, y_text + th + 8), color, -1)
    cv2.putText(
        out,
        text,
        (x1 + 4, y_text + th + 4),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )
    return out


In [ ]:
%%writefile scripts/__init__.py
"""Command-line helper scripts."""


In [ ]:
%%writefile scripts/build_cloud_crops.py
from __future__ import annotations

import argparse
from pathlib import Path

import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO

from cloud_chaser.data.gcd import IMAGE_EXTENSIONS


def crop_dataset(input_root: Path, output_root: Path, detector_weights: str, conf: float, padding: int) -> None:
    model = YOLO(detector_weights)
    paths = sorted(p for p in input_root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
    for path in tqdm(paths, desc="Cropping cloud regions"):
        image_bgr = cv2.imread(str(path))
        if image_bgr is None:
            continue
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        result = model.predict(image_rgb, conf=conf, retina_masks=True, verbose=False)[0]
        if result.boxes is None or len(result.boxes) == 0:
            rel = path.relative_to(input_root)
            target = output_root / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(target), image_bgr)
            continue
        h, w = image_bgr.shape[:2]
        masks = result.masks.data.detach().cpu().numpy() if result.masks is not None else []
        boxes = result.boxes.xyxy.detach().cpu().numpy()
        for idx, box in enumerate(boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            x1 = max(0, x1 - padding)
            y1 = max(0, y1 - padding)
            x2 = min(w, x2 + padding)
            y2 = min(h, y2 + padding)
            crop = image_bgr.copy()
            if len(masks):
                mask = masks[idx]
                if mask.shape != (h, w):
                    mask = cv2.resize(mask.astype("float32"), (w, h), interpolation=cv2.INTER_NEAREST)
                crop[mask <= 0.5] = 0
            crop = crop[y1:y2, x1:x2]
            rel = path.relative_to(input_root)
            target = output_root / rel.parent / f"{rel.stem}_cloud{idx}{rel.suffix}"
            target.parent.mkdir(parents=True, exist_ok=True)
            if np.prod(crop.shape[:2]) > 0:
                cv2.imwrite(str(target), crop)


def main() -> None:
    parser = argparse.ArgumentParser(description="Build classifier crop folders from detector masks.")
    parser.add_argument("--input-root", required=True, type=Path)
    parser.add_argument("--output-root", required=True, type=Path)
    parser.add_argument("--detector-weights", required=True)
    parser.add_argument("--conf", type=float, default=0.25)
    parser.add_argument("--padding", type=int, default=12)
    args = parser.parse_args()
    crop_dataset(args.input_root, args.output_root, args.detector_weights, args.conf, args.padding)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/prepare_ade20k_yolo.py
from __future__ import annotations

import argparse

from cloud_chaser.config import load_config
from cloud_chaser.data.ade20k import prepare_ade20k_yolo


def main() -> None:
    parser = argparse.ArgumentParser(description="Convert ADE20K cloud/sky masks to YOLO segmentation format.")
    parser.add_argument("--config", default="configs/default.yaml")
    args = parser.parse_args()
    cfg = load_config(args.config)
    data_cfg = cfg["data"]
    path = prepare_ade20k_yolo(
        root=data_cfg["ade20k_root"],
        output_dir=data_cfg["prepared_seg_dir"],
        class_names=data_cfg["ade_classes"],
        fallback_class_ids=data_cfg.get("ade_fallback_class_ids", []),
        min_mask_area=data_cfg["min_mask_area"],
    )
    print(f"wrote={path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/prepare_swimseg_yolo.py
from __future__ import annotations

import argparse

from cloud_chaser.config import load_config
from cloud_chaser.data.swimseg import prepare_swimseg_yolo


def main() -> None:
    parser = argparse.ArgumentParser(description="Convert SWIMSEG cloud masks to YOLO segmentation format.")
    parser.add_argument("--config", default="configs/default.yaml")
    args = parser.parse_args()
    cfg = load_config(args.config)
    data_cfg = cfg["data"]
    path = prepare_swimseg_yolo(
        root=data_cfg["swimseg_root"],
        output_dir=data_cfg["prepared_seg_dir"],
        val_fraction=data_cfg.get("seg_val_fraction", 0.1),
        test_fraction=data_cfg.get("seg_test_fraction", 0.1),
        seed=cfg["project"]["seed"],
        min_mask_area=data_cfg.get("min_mask_area", 96),
        invert_masks=data_cfg.get("swimseg_invert_masks", False),
    )
    print(f"wrote={path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile configs/default.yaml
project:
  name: cloud-chaser
  seed: 42
  device: cuda
  output_dir: /kaggle/working/cloud-chaser/runs

data:
  swimseg_root: /kaggle/input/skyimage-dataset/swimseg-2
  gcd_root: /kaggle/input/gcd/GCD
  tjnu_root: /kaggle/input/tjnu/TJNU
  prepared_seg_dir: /kaggle/working/cloud-chaser/data/processed/swimseg_cloud_yolo
  num_workers: 2
  image_size: 224
  val_fraction: 0.15
  seg_val_fraction: 0.1
  seg_test_fraction: 0.1
  classification_classes:
    - 1_cumulus
    - 2_altocumulus
    - 3_cirrus
    - 4_clearsky
    - 5_stratocumulus
    - 6_cumulonimbus
    - 7_mixed
  min_mask_area: 96
  swimseg_invert_masks: false

augmentation:
  random_shadow_p: 0.25
  gaussian_blur_p: 0.20
  hflip_p: 0.50
  vflip_p: 0.15

detector:
  model: yolo11s-seg.pt
  epochs: 80
  imgsz: 640
  batch: 8
  patience: 20
  lr0: 0.01
  weight_decay: 0.0005
  conf: 0.25
  iou: 0.60
  half: true

ssl:
  method: simclr
  backbone: resnet50
  projection_dim: 128
  hidden_dim: 2048
  batch_size: 64
  epochs: 100
  lr: 0.0003
  weight_decay: 0.000001
  temperature: 0.2

classifier:
  backbone: resnet50
  batch_size: 32
  epochs: 40
  lr: 0.0001
  weight_decay: 0.0001
  dropout: 0.2
  amp: true
  freeze_backbone_epochs: 2
  ssl_checkpoint: null
  checkpoint: /kaggle/working/cloud-chaser/runs/classifier/best.pt

inference:
  detector_weights: /kaggle/working/cloud-chaser/runs/detector/weights/best.pt
  classifier_weights: /kaggle/working/cloud-chaser/runs/classifier/best.pt
  output_dir: /kaggle/working/cloud-chaser/runs/inference
  crop_padding: 12
  mask_alpha: 0.42


## Install Dependencies

In [ ]:
import torch, sys
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


In [ ]:
import subprocess
subprocess.run(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], check=True)


## Configure Dataset Paths

Attach `antigs/skyimage-dataset` and your GCD Kaggle dataset, then run this cell. It auto-discovers common folder names under `/kaggle/input`.

In [ ]:
from pathlib import Path
import yaml
import torch

cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
KAGGLE_INPUT = Path('/kaggle/input')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def image_count(path):
    return sum(1 for p in Path(path).rglob('*') if p.suffix.lower() in IMAGE_EXTS)

def find_existing_dir(candidates, contains=None):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return str(path)
    if KAGGLE_INPUT.exists() and contains:
        matches = [p for p in KAGGLE_INPUT.rglob('*') if p.is_dir() and contains.lower() in p.name.lower()]
        if matches:
            return str(sorted(matches, key=lambda p: (len(p.parts), str(p)))[0])
    return str(candidates[0])

def find_swimseg_root():
    explicit = [
        Path('/kaggle/input/skyimage-dataset/swimseg-2'),
        Path('/kaggle/input/skyimage-dataset/SWIMSEG-2'),
        Path('/kaggle/input/datasets/antigs/skyimage-dataset/swimseg-2'),
        Path('/kaggle/input/datasets/antigs/skyimage-dataset/SWIMSEG-2'),
        Path('/kaggle/input/datasets/skyimage-dataset/swimseg-2'),
    ]
    for path in explicit:
        if path.exists():
            return str(path)
    candidates = []
    if KAGGLE_INPUT.exists():
        for path in KAGGLE_INPUT.rglob('*'):
            if not path.is_dir():
                continue
            text = str(path).lower()
            if 'swimseg' not in text:
                continue
            count = image_count(path)
            if count:
                candidates.append((count, path))
    if candidates:
        return str(sorted(candidates, key=lambda item: (-item[0], len(item[1].parts), str(item[1])))[0][1])
    return ''

def has_split_dir(path, split_names):
    return any((path / name).exists() and (path / name).is_dir() for name in split_names)

def is_not_segmentation_dataset(path):
    text = str(path).lower()
    banned = ['swimseg', 'skyimage', 'sky-image', 'segmentation', 'mask']
    return not any(token in text for token in banned)

def find_gcd_root():
    explicit = [
        Path('/kaggle/input/gcd/GCD'),
        Path('/kaggle/input/gcd'),
        Path('/kaggle/input/global-cloud-dataset/GCD'),
        Path('/kaggle/input/global-cloud-dataset'),
        Path('/kaggle/input/datasets/alopixalopix/gcd-dataset/GCD'),
        Path('/kaggle/input/datasets/alopixalopix/gcd-dataset'),
    ]
    for path in explicit:
        if path.exists() and is_not_segmentation_dataset(path) and has_split_dir(path, ['train', 'Train', 'training', 'Training']):
            return str(path)
    candidates = []
    if KAGGLE_INPUT.exists():
        for path in KAGGLE_INPUT.rglob('*'):
            if not path.is_dir() or not is_not_segmentation_dataset(path):
                continue
            class_root = None
            if has_split_dir(path, ['train', 'Train', 'training', 'Training']):
                class_root = next((path / n for n in ['train', 'Train', 'training', 'Training'] if (path / n).exists()), None)
            else:
                class_root = path
            class_dirs = [p for p in class_root.iterdir() if p.is_dir()] if class_root and class_root.exists() else []
            names = ' '.join(p.name.lower().replace('_', '') for p in class_dirs)
            cloud_hits = sum(1 for token in ['cumulus', 'altocumulus', 'cirrus', 'clearsky', 'stratocumulus', 'cumulonimbus', 'mixed'] if token in names)
            count = image_count(class_root) if class_root else 0
            if len(class_dirs) >= 2 and count and cloud_hits >= 2:
                candidates.append((cloud_hits, count, path))
    if candidates:
        return str(sorted(candidates, key=lambda item: (-item[0], -item[1], len(item[2].parts), str(item[2])))[0][2])
    return ''

cfg['data']['swimseg_root'] = find_swimseg_root()
cfg['data']['gcd_root'] = find_gcd_root()
cfg['data']['tjnu_root'] = find_existing_dir(['/kaggle/input/tjnu/TJNU', '/kaggle/input/tjnu'], contains='TJNU')
cfg['data']['prepared_seg_dir'] = '/kaggle/working/cloud-chaser/data/processed/swimseg_cloud_yolo'
cfg['project']['output_dir'] = '/kaggle/working/cloud-chaser/runs'
cfg['project']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg['inference']['detector_weights'] = '/kaggle/working/cloud-chaser/runs/detector/weights/best.pt'
cfg['inference']['classifier_weights'] = '/kaggle/working/cloud-chaser/runs/classifier/best.pt'
cfg['classifier']['checkpoint'] = '/kaggle/working/cloud-chaser/runs/classifier/best.pt'

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg_path.read_text())
for label, key in [('SWIMSEG', 'swimseg_root'), ('GCD', 'gcd_root'), ('TJNU', 'tjnu_root')]:
    value = cfg['data'][key]
    path = Path(value) if value else Path('__missing__')
    print(f'{label}: {value or "NOT FOUND"} exists={path.exists()}')

if not cfg['data']['swimseg_root'] or not Path(cfg['data']['swimseg_root']).exists():
    print('\nAvailable /kaggle/input folders:')
    for path in sorted(KAGGLE_INPUT.glob('*')) if KAGGLE_INPUT.exists() else []:
        print(' -', path)
    print('\nFirst nested folders:')
    nested = [p for p in sorted(KAGGLE_INPUT.rglob('*')) if p.is_dir()][:120] if KAGGLE_INPUT.exists() else []
    for path in nested:
        print(' -', path)
    raise FileNotFoundError('Attach the Kaggle Sky-Image Dataset, or update data.swimseg_root above.')


## Dataset Tree Debug

Run this if path discovery fails or if the SWIMSEG/GCD folder names differ.

In [ ]:
from pathlib import Path
for root in sorted(Path('/kaggle/input').glob('*')):
    print(root)
    for child in sorted(root.rglob('*'))[:80]:
        print('  ', child.relative_to(root))


## GCD Dataset Sanity Check


In [ ]:
from pathlib import Path
import yaml
cfg = yaml.safe_load(Path('configs/default.yaml').read_text())
gcd_value = cfg['data'].get('gcd_root')
print('GCD root:', gcd_value or 'NOT FOUND')
if not gcd_value:
    print('No GCD classification dataset was detected. Attach GCD or set data.gcd_root manually.')
else:
    gcd_root = Path(gcd_value)
    print('exists=', gcd_root.exists())
    for split_name in ['train', 'Train', 'training', 'Training', 'test', 'Test', 'val', 'validation']:
        split_dir = gcd_root / split_name
        if split_dir.exists():
            print('\n', split_dir)
            for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
                count = sum(1 for p in class_dir.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
                print(' ', class_dir.name, count)


## Prepare SWIMSEG-2 for YOLO Segmentation

In [ ]:
import subprocess
subprocess.run(['python', '-m', 'scripts.prepare_swimseg_yolo', '--config', 'configs/default.yaml'], check=True)


## Train Detector

In [ ]:
from pathlib import Path
import subprocess

best = Path('/kaggle/working/cloud-chaser/runs/detector/weights/best.pt')
last = Path('/kaggle/working/cloud-chaser/runs/detector/weights/last.pt')
if best.exists():
    print(f'Skipping detector training; found checkpoint: {best}')
elif last.exists():
    print(f'Detector last checkpoint exists but best.pt is missing: {last}')
    print('Using last.pt as detector best checkpoint for downstream steps.')
    best.parent.mkdir(parents=True, exist_ok=True)
    best.write_bytes(last.read_bytes())
else:
    assert Path('train.py').exists(), 'train.py was not written; rerun notebook from the top.'
    subprocess.run(['python', 'train.py', 'detector', '--config', 'configs/default.yaml'], check=True)


## Optional: SimCLR Pretraining on TJNU

In [ ]:
from pathlib import Path
import subprocess
import yaml

cfg = yaml.safe_load(Path('configs/default.yaml').read_text())
tjnu_root = Path(cfg['data']['tjnu_root'])
ssl_best = Path('/kaggle/working/cloud-chaser/runs/ssl/best.pt')
if ssl_best.exists():
    print(f'Skipping optional SimCLR pretraining; found checkpoint: {ssl_best}')
elif tjnu_root.exists():
    subprocess.run(['python', 'train.py', 'ssl', '--config', 'configs/default.yaml'], check=True)
else:
    print(f'Skipping optional SimCLR pretraining: TJNU dataset not found at {tjnu_root}')
    print('Classifier training will continue from ImageNet-pretrained backbone instead.')


## Train GCD Classifier

In [ ]:
from pathlib import Path
import subprocess
import yaml

cfg = yaml.safe_load(Path('configs/default.yaml').read_text())
classifier_best = Path('/kaggle/working/cloud-chaser/runs/classifier/best.pt')
print('Configured GCD root:', cfg['data'].get('gcd_root') or 'NOT FOUND')
print('Available Kaggle inputs:')
for path in sorted(Path('/kaggle/input').glob('*')):
    image_count = sum(1 for p in path.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
    print(' -', path, 'images=', image_count)

if classifier_best.exists():
    print(f'Skipping classifier training; found checkpoint: {classifier_best}')
else:
    gcd_root = cfg['data'].get('gcd_root')
    if not gcd_root or not Path(gcd_root).exists():
        print('\nSkipping classifier training: no GCD classification dataset is attached.')
        print('Attach the GCD dataset or set cfg["data"]["gcd_root"] to a folder with train/<class> images.')
    else:
        subprocess.run(['python', 'train.py', 'classifier', '--config', 'configs/default.yaml'], check=True)


## Evaluate

In [ ]:
from pathlib import Path
import subprocess

best_detector = Path('/kaggle/working/cloud-chaser/runs/detector/weights/best.pt')
best_classifier = Path('/kaggle/working/cloud-chaser/runs/classifier/best.pt')

if best_detector.exists():
    print(f'Evaluating detector checkpoint: {best_detector}')
    result = subprocess.run(['python', 'train.py', 'eval-detector', '--config', 'configs/default.yaml'], check=False)
    if result.returncode:
        print('Detector evaluation failed, but the detector checkpoint exists. Continuing notebook run.')
else:
    print('Skipping detector evaluation: missing', best_detector)

if best_classifier.exists():
    print(f'Evaluating classifier checkpoint: {best_classifier}')
    result = subprocess.run(['python', 'train.py', 'eval-classifier', '--config', 'configs/default.yaml'], check=False)
    if result.returncode:
        print('Classifier evaluation failed, but the classifier checkpoint exists. Continuing notebook run.')
else:
    print('Skipping classifier evaluation: missing', best_classifier)


## Inference Example

Change `IMAGE_PATH` to a real image in `/kaggle/input` or `/kaggle/working`.

In [ ]:
from pathlib import Path
import subprocess
from IPython.display import Image, display

IMAGE_PATH = Path('/kaggle/input/example-outdoor-scene/image.jpg')
OUTPUT_PATH = Path('/kaggle/working/cloud_chaser_prediction.jpg')
required = [
    Path('/kaggle/working/cloud-chaser/runs/detector/weights/best.pt'),
    Path('/kaggle/working/cloud-chaser/runs/classifier/best.pt'),
    IMAGE_PATH,
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print('Skipping two-stage inference because required files are missing:', missing)
else:
    subprocess.run(['python', 'inference.py', '--config', 'configs/default.yaml', '--image', str(IMAGE_PATH), '--output', str(OUTPUT_PATH)], check=True)
    display(Image(filename=str(OUTPUT_PATH)))


## Export Artifacts

In [ ]:
from pathlib import Path
import subprocess

best_detector = Path('/kaggle/working/cloud-chaser/runs/detector/weights/best.pt')
best_classifier = Path('/kaggle/working/cloud-chaser/runs/classifier/best.pt')
classifier_onnx = Path('/kaggle/working/classifier.onnx')
classifier_ts = Path('/kaggle/working/classifier.torchscript')

if best_detector.exists():
    print('Exporting detector if not already exported by Ultralytics.')
    subprocess.run(['python', 'export.py', 'detector', '--config', 'configs/default.yaml', '--format', 'onnx'], check=False)
else:
    print('Skipping detector export: missing', best_detector)

if best_classifier.exists():
    if classifier_onnx.exists():
        print('Skipping classifier ONNX export; found', classifier_onnx)
    else:
        subprocess.run(['python', 'export.py', 'classifier', '--config', 'configs/default.yaml', '--format', 'onnx', '--output', str(classifier_onnx)], check=False)
    if classifier_ts.exists():
        print('Skipping classifier TorchScript export; found', classifier_ts)
    else:
        subprocess.run(['python', 'export.py', 'classifier', '--config', 'configs/default.yaml', '--format', 'torchscript', '--output', str(classifier_ts)], check=False)
else:
    print('Skipping classifier export: missing', best_classifier)
